In [1]:
!pip install dagshub mlflow -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.2/49.2 kB 1.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 2.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 273.1/273.1 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 75.1 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 76.0 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 51.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.2/68.2 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.4/208.4 kB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.2/132.2 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
import os
import gc
import numpy as np
import pandas as pd
import mlflow
import mlflow.xgboost
import dagshub
import xgboost as xgb
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.metrics import roc_auc_score
from sklearn.pipeline import Pipeline
import warnings
warnings.filterwarnings('ignore')

dagshub.init(
    repo_owner="sophyrise",
    repo_name='ieee-cis-fraud-detection',
    mlflow=True
)

mlflow.set_experiment("XGBoost")
print("✅ MLflow tracking URI:", mlflow.get_tracking_uri())
print("✅ XGBoost version:", xgb.__version__)

❗❗❗ AUTHORIZATION REQUIRED ❗❗❗

Output()



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=d6c36292-d973-4790-baf7-1263841e55e9&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=accfbba469294158c73b6e78aba18a0e64e84b776f2589b3b40d3e1b762b5490




Accessing as sophyrise

Initialized MLflow to track repo "sophyrise/ieee-cis-fraud-detection"

Repository sophyrise/ieee-cis-fraud-detection initialized!

✅ MLflow tracking URI: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow
✅ XGBoost version: 3.2.0


# **Cleaning**

In [3]:
with mlflow.start_run(run_name="XGBoost_Cleaning"):
    DATA_DIR = "/kaggle/input/competitions/ieee-fraud-detection"
    TXN_MISSING_THRESHOLD   = 0.80
    ID_MISSING_THRESHOLD    = 0.95
    NEAR_CONSTANT_THRESHOLD = 0.99

    train_txn = pd.read_csv(f"{DATA_DIR}/train_transaction.csv")
    train_id  = pd.read_csv(f"{DATA_DIR}/train_identity.csv")
    test_txn  = pd.read_csv(f"{DATA_DIR}/test_transaction.csv")
    test_id   = pd.read_csv(f"{DATA_DIR}/test_identity.csv")

    test_id.columns  = [c.replace('-', '_') for c in test_id.columns]
    train_id.columns = [c.replace('-', '_') for c in train_id.columns]

    train = train_txn.merge(train_id, on="TransactionID", how="left")
    test  = test_txn.merge(test_id,  on="TransactionID", how="left")

    y_train_xgb = train["isFraud"].copy()
    X_train_xgb = train.drop(columns=["isFraud", "TransactionID"])
    X_test_xgb  = test.drop(columns=["TransactionID"])

    del train, test, train_txn, train_id, test_txn, test_id
    gc.collect()

    id_like_cols  = [c for c in X_train_xgb.columns if c.startswith("id_") or c in ["DeviceType", "DeviceInfo"]]
    txn_like_cols = [c for c in X_train_xgb.columns if c not in id_like_cols]

    missing_ratio = X_train_xgb.isnull().mean()
    drop_txn      = [c for c in txn_like_cols if missing_ratio[c] > TXN_MISSING_THRESHOLD]
    drop_id       = [c for c in id_like_cols  if missing_ratio[c] > ID_MISSING_THRESHOLD]
    drop_missing  = drop_txn + drop_id

    X_train_xgb = X_train_xgb.drop(columns=drop_missing)
    X_test_xgb  = X_test_xgb.drop(columns=[c for c in drop_missing if c in X_test_xgb.columns])

    near_constant_cols = []
    for col in X_train_xgb.columns:
        top_freq = X_train_xgb[col].value_counts(dropna=False, normalize=True).iloc[0]
        if top_freq > NEAR_CONSTANT_THRESHOLD:
            near_constant_cols.append(col)

    X_train_xgb = X_train_xgb.drop(columns=near_constant_cols)
    X_test_xgb  = X_test_xgb.drop(columns=[c for c in near_constant_cols if c in X_test_xgb.columns])

    for col in X_train_xgb.columns:
        if col not in X_test_xgb.columns:
            X_test_xgb[col] = np.nan
    X_test_xgb = X_test_xgb[X_train_xgb.columns]

    mlflow.log_param("txn_missing_threshold",   TXN_MISSING_THRESHOLD)
    mlflow.log_param("id_missing_threshold",    ID_MISSING_THRESHOLD)
    mlflow.log_param("near_constant_threshold", NEAR_CONSTANT_THRESHOLD)
    mlflow.log_metric("train_rows",             int(X_train_xgb.shape[0]))
    mlflow.log_metric("test_rows",              int(X_test_xgb.shape[0]))
    mlflow.log_metric("features_after_cleaning",int(X_train_xgb.shape[1]))
    mlflow.log_metric("dropped_high_missing",   len(drop_missing))
    mlflow.log_metric("dropped_near_constant",  len(near_constant_cols))
    mlflow.log_metric("fraud_rate",             float(y_train_xgb.mean()))

    print(f"X_train: {X_train_xgb.shape}")
    print(f"X_test:  {X_test_xgb.shape}")
    print(f"Fraud rate: {y_train_xgb.mean():.4f}")

    X_train_clean_xgb = X_train_xgb
    X_test_clean_xgb  = X_test_xgb
    y_train_clean_xgb = y_train_xgb

X_train: (590540, 353)
X_test:  (506691, 353)
Fraud rate: 0.0350
🏃 View run XGBoost_Cleaning at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/4/runs/d140a40265b244ef96958c164d0704c4
🧪 View experiment at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/4


# Feature Engineering

In [4]:
with mlflow.start_run(run_name="XGBoost_FeatureEngineering"):
    X_train = X_train_clean_xgb.copy()
    X_test  = X_test_clean_xgb.copy()
    y_train = y_train_clean_xgb.copy()

    # log transform — TransactionAmt is heavily right-skewed
    X_train["TransactionAmt_log"] = np.log1p(X_train["TransactionAmt"].clip(lower=0))
    X_test["TransactionAmt_log"]  = np.log1p(X_test["TransactionAmt"].clip(lower=0))

    # cyclic hour encoding — fraud has strong time-of-day pattern
    X_train["hour_sin"] = np.sin(2 * np.pi * ((X_train["TransactionDT"] // 3600) % 24) / 24)
    X_train["hour_cos"] = np.cos(2 * np.pi * ((X_train["TransactionDT"] // 3600) % 24) / 24)
    X_test["hour_sin"]  = np.sin(2 * np.pi * ((X_test["TransactionDT"]  // 3600) % 24) / 24)
    X_test["hour_cos"]  = np.cos(2 * np.pi * ((X_test["TransactionDT"]  // 3600) % 24) / 24)

    X_train = X_train.drop(columns=["TransactionDT"], errors="ignore")
    X_test  = X_test.drop(columns=["TransactionDT"],  errors="ignore")

    # ordinal encode categoricals
    # XGBoost handles ordinal-encoded cats fine — no one-hot needed
    cat_cols = X_train.select_dtypes(include=["object"]).columns.tolist()
    num_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()

    for c in cat_cols:
        mapping = {v: i for i, v in enumerate(X_train[c].astype(str).unique())}
        X_train[c] = X_train[c].astype(str).map(mapping).fillna(-1).astype(np.int32)
        X_test[c]  = X_test[c].astype(str).map(mapping).fillna(-1).astype(np.int32)

    X_test = X_test.reindex(columns=X_train.columns, fill_value=-1)

    mlflow.log_param("cat_encoding",     "ordinal_train_mapping")
    mlflow.log_param("amt_log_transform", True)
    mlflow.log_param("cyclic_time_encoding", True)
    mlflow.log_metric("features_after_fe",   int(X_train.shape[1]))
    mlflow.log_metric("cat_cols_encoded",    len(cat_cols))

    print(f"X_train_fe: {X_train.shape}")
    print(f"X_test_fe:  {X_test.shape}")

    X_train_fe_xgb = X_train
    X_test_fe_xgb  = X_test
    y_train_fe_xgb = y_train

X_train_fe: (590540, 355)
X_test_fe:  (506691, 355)
🏃 View run XGBoost_FeatureEngineering at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/4/runs/c7c81802b5574bcda7c8993b0ba869e1
🧪 View experiment at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/4


# Feature Selection

In [5]:
with mlflow.start_run(run_name="XGBoost_FeatureSelection"):
    X_train = X_train_fe_xgb.copy()
    X_test  = X_test_fe_xgb.copy()

    X_train = X_train.replace([np.inf, -np.inf], np.nan).fillna(0)
    X_test  = X_test.replace([np.inf, -np.inf],  np.nan).fillna(0)

    # drop constant columns
    nu = X_train.nunique(dropna=False)
    const_cols = nu[nu <= 1].index.tolist()
    X_train = X_train.drop(columns=const_cols, errors="ignore")
    X_test  = X_test.drop(columns=const_cols,  errors="ignore")

    # drop highly correlated features — sampled for speed
    CORR_THRESHOLD = 0.98
    num_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()
    sample_n = min(120_000, len(X_train))
    idx  = np.random.RandomState(42).choice(len(X_train), size=sample_n, replace=False)
    corr = X_train.iloc[idx][num_cols].corr().abs()
    upper     = corr.where(np.triu(np.ones(corr.shape, dtype=bool), k=1))
    drop_corr = [c for c in upper.columns if (upper[c] > CORR_THRESHOLD).any()]

    X_train = X_train.drop(columns=drop_corr, errors="ignore")
    X_test  = X_test.drop(columns=drop_corr,  errors="ignore")
    X_test  = X_test.reindex(columns=X_train.columns, fill_value=0)

    mlflow.log_param("corr_threshold",     CORR_THRESHOLD)
    mlflow.log_metric("dropped_const",     len(const_cols))
    mlflow.log_metric("dropped_high_corr", len(drop_corr))
    mlflow.log_metric("features_after_fs", int(X_train.shape[1]))

    print(f"X_train_fs: {X_train.shape}")
    print(f"X_test_fs:  {X_test.shape}")

    X_train_final_xgb = X_train
    X_test_final_xgb  = X_test

X_train_fs: (590540, 301)
X_test_fs:  (506691, 301)
🏃 View run XGBoost_FeatureSelection at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/4/runs/85eca9a96cb0430aaa14dfdfa6c6f27f
🧪 View experiment at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/4


# Training

In [6]:
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train_final_xgb, y_train_fe_xgb,
    test_size=0.2, random_state=42, stratify=y_train_fe_xgb
)

neg = int((y_tr == 0).sum())
pos = int((y_tr == 1).sum())
spw = round(neg / pos, 2)
print(f"Train size: {X_tr.shape} | Val size: {X_val.shape}")
print(f"scale_pos_weight = {spw}  (neg/pos = {neg}/{pos})")

Train size: (472432, 301) | Val size: (118108, 301)
scale_pos_weight = 27.58  (neg/pos = 455902/16530)


In [7]:
with mlflow.start_run(run_name="XGB_Baseline"):
    mlflow.log_param("n_estimators",     100)
    mlflow.log_param("max_depth",        6)
    mlflow.log_param("learning_rate",    0.3)
    mlflow.log_param("scale_pos_weight", 1)
    mlflow.log_param("tree_method",      "hist")
    mlflow.log_param("device",           "cuda")
    mlflow.log_param("note",             "default params, no class balancing")

    clf = xgb.XGBClassifier(
        n_estimators=100, max_depth=6, learning_rate=0.3,
        scale_pos_weight=1, tree_method="hist", device="cuda",
        eval_metric="auc", random_state=42
    )
    clf.fit(X_tr, y_tr)

    train_auc = roc_auc_score(y_tr,  clf.predict_proba(X_tr)[:, 1])
    val_auc   = roc_auc_score(y_val, clf.predict_proba(X_val)[:, 1])
    gap = train_auc - val_auc

    mlflow.log_metric("train_auc",   round(train_auc, 5))
    mlflow.log_metric("val_auc",     round(val_auc,   5))
    mlflow.log_metric("overfit_gap", round(gap, 5))

    print(f"[Baseline] Train: {train_auc:.4f} | Val: {val_auc:.4f} | Gap: {gap:.4f}")
    print("  -> default lr=0.3, no class balancing.")
    print(f"  -> Gap {gap:.4f}: {'OVERFIT' if gap > 0.02 else 'acceptable'}")

[Baseline] Train: 0.9563 | Val: 0.9388 | Gap: 0.0175
  -> default lr=0.3, no class balancing.
  -> Gap 0.0175: acceptable
🏃 View run XGB_Baseline at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/4/runs/715a572a88044f94822f3a1733269265
🧪 View experiment at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/4


In [10]:
for n_est in [100, 300]:
    with mlflow.start_run(run_name=f"XGB_nestimators_{n_est}"):
        mlflow.log_param("n_estimators",     n_est)
        mlflow.log_param("max_depth",        6)
        mlflow.log_param("learning_rate",    0.1)
        mlflow.log_param("scale_pos_weight", spw)
        mlflow.log_param("tree_method",      "hist")
        mlflow.log_param("device",           "cuda")

        clf = xgb.XGBClassifier(
            n_estimators=n_est, max_depth=6, learning_rate=0.1,
            scale_pos_weight=spw, tree_method="hist", device="cuda",
            eval_metric="auc", random_state=42
        )
        clf.fit(X_tr, y_tr)

        train_auc = roc_auc_score(y_tr,  clf.predict_proba(X_tr)[:, 1])
        val_auc   = roc_auc_score(y_val, clf.predict_proba(X_val)[:, 1])
        gap = train_auc - val_auc

        mlflow.log_metric("train_auc",   round(train_auc, 5))
        mlflow.log_metric("val_auc",     round(val_auc,   5))
        mlflow.log_metric("overfit_gap", round(gap, 5))

        status = "OVERFIT" if gap > 0.025 else ("UNDERFIT" if val_auc < 0.90 else "OK")
        print(f"  n_estimators={n_est:<4} | Train: {train_auc:.4f} | Val: {val_auc:.4f} | Gap: {gap:.4f} [{status}]")

  n_estimators=100  | Train: 0.9370 | Val: 0.9227 | Gap: 0.0143 [OK]
🏃 View run XGB_nestimators_100 at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/4/runs/e74fc2afd6bf4c758a6944e293ce4233
🧪 View experiment at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/4
  n_estimators=300  | Train: 0.9686 | Val: 0.9458 | Gap: 0.0228 [OK]
🏃 View run XGB_nestimators_300 at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/4/runs/4c09ab6c4e7d4734a66ebdc4897f9ff0
🧪 View experiment at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/4


In [11]:
depth_results = []
for depth in [3, 4, 6, 8, 10]:
    with mlflow.start_run(run_name=f"XGB_depth_{depth}"):
        mlflow.log_param("n_estimators",     300)
        mlflow.log_param("max_depth",        depth)
        mlflow.log_param("learning_rate",    0.1)
        mlflow.log_param("scale_pos_weight", spw)
        mlflow.log_param("tree_method",      "hist")
        mlflow.log_param("device",           "cuda")

        clf = xgb.XGBClassifier(
            n_estimators=300, max_depth=depth, learning_rate=0.1,
            scale_pos_weight=spw, tree_method="hist", device="cuda",
            eval_metric="auc", random_state=42
        )
        clf.fit(X_tr, y_tr)

        train_auc = roc_auc_score(y_tr,  clf.predict_proba(X_tr)[:, 1])
        val_auc   = roc_auc_score(y_val, clf.predict_proba(X_val)[:, 1])
        gap = train_auc - val_auc

        mlflow.log_metric("train_auc",   round(train_auc, 5))
        mlflow.log_metric("val_auc",     round(val_auc,   5))
        mlflow.log_metric("overfit_gap", round(gap, 5))

        # explicit diagnosis for each depth
        if gap > 0.03:
            diagnosis = "OVERFIT — depth too large, model memorising train set"
        elif val_auc < 0.91:
            diagnosis = "UNDERFIT — depth too shallow, model lacks capacity"
        else:
            diagnosis = "OK"

        depth_results.append({"max_depth": depth, "train_auc": train_auc,
                               "val_auc": val_auc, "gap": gap})
        print(f"  depth={depth:<2} | Train: {train_auc:.4f} | Val: {val_auc:.4f} | Gap: {gap:.4f} — {diagnosis}")

depth_df  = pd.DataFrame(depth_results).set_index("max_depth")
best_depth = int(depth_df["val_auc"].idxmax())
print(f"\n-> Best depth by val AUC: {best_depth}")
print(depth_df.to_string())

  depth=3  | Train: 0.9162 | Val: 0.9076 | Gap: 0.0086 — UNDERFIT — depth too shallow, model lacks capacity
🏃 View run XGB_depth_3 at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/4/runs/a58434ef8b1d4f5b82dacdd7bf09037d
🧪 View experiment at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/4
  depth=4  | Train: 0.9349 | Val: 0.9221 | Gap: 0.0128 — OK
🏃 View run XGB_depth_4 at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/4/runs/28153bb8b53148a6949b357669ed03c1
🧪 View experiment at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/4
  depth=6  | Train: 0.9686 | Val: 0.9458 | Gap: 0.0228 — OK
🏃 View run XGB_depth_6 at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/4/runs/ac2f1693d50e49e58d35061049c803dc
🧪 View experiment at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/4
  depth=8  | Train: 0.9889 | Val: 0.9582 | 

In [12]:
for subsample, colsample in [(1.0, 1.0), (0.8, 0.8), (0.7, 0.7)]:
    label = f"sub{subsample}_col{colsample}"
    with mlflow.start_run(run_name=f"XGB_sampling_{label}"):
        mlflow.log_param("n_estimators",     300)
        mlflow.log_param("max_depth",        best_depth)
        mlflow.log_param("learning_rate",    0.1)
        mlflow.log_param("subsample",        subsample)
        mlflow.log_param("colsample_bytree", colsample)
        mlflow.log_param("scale_pos_weight", spw)
        mlflow.log_param("tree_method",      "hist")
        mlflow.log_param("device",           "cuda")

        clf = xgb.XGBClassifier(
            n_estimators=300, max_depth=best_depth, learning_rate=0.1,
            subsample=subsample, colsample_bytree=colsample,
            scale_pos_weight=spw, tree_method="hist", device="cuda",
            eval_metric="auc", random_state=42
        )
        clf.fit(X_tr, y_tr)

        train_auc = roc_auc_score(y_tr,  clf.predict_proba(X_tr)[:, 1])
        val_auc   = roc_auc_score(y_val, clf.predict_proba(X_val)[:, 1])
        gap = train_auc - val_auc

        mlflow.log_metric("train_auc",   round(train_auc, 5))
        mlflow.log_metric("val_auc",     round(val_auc,   5))
        mlflow.log_metric("overfit_gap", round(gap, 5))

        print(f"  {label} | Train: {train_auc:.4f} | Val: {val_auc:.4f} | Gap: {gap:.4f}")

# pick best from output
best_subsample = 0.8
best_colsample = 0.8

  sub1.0_col1.0 | Train: 0.9974 | Val: 0.9636 | Gap: 0.0338
🏃 View run XGB_sampling_sub1.0_col1.0 at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/4/runs/97ff362e712d426e81d5a8db669b4c92
🧪 View experiment at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/4
  sub0.8_col0.8 | Train: 0.9995 | Val: 0.9657 | Gap: 0.0338
🏃 View run XGB_sampling_sub0.8_col0.8 at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/4/runs/8657bca51c3c418d84fd3355b4a19a2a
🧪 View experiment at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/4
  sub0.7_col0.7 | Train: 0.9996 | Val: 0.9655 | Gap: 0.0341
🏃 View run XGB_sampling_sub0.7_col0.7 at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/4/runs/5ee2e9dffbde4c64aeb1d477e35f1948
🧪 View experiment at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/4


In [13]:
# L1 (reg_alpha) and L2 (reg_lambda) reduce overfitting
# testing combinations — high gap from depth experiments makes this important
reg_results = []
for alpha, lam in [(0, 1), (0.1, 1), (1, 1), (0, 5), (1, 5)]:
    label = f"a{alpha}_l{lam}"
    with mlflow.start_run(run_name=f"XGB_reg_{label}"):
        mlflow.log_param("n_estimators",     300)
        mlflow.log_param("max_depth",        best_depth)
        mlflow.log_param("learning_rate",    0.1)
        mlflow.log_param("subsample",        best_subsample)
        mlflow.log_param("colsample_bytree", best_colsample)
        mlflow.log_param("reg_alpha",        alpha)
        mlflow.log_param("reg_lambda",       lam)
        mlflow.log_param("scale_pos_weight", spw)
        mlflow.log_param("tree_method",      "hist")
        mlflow.log_param("device",           "cuda")

        clf = xgb.XGBClassifier(
            n_estimators=300, max_depth=best_depth, learning_rate=0.1,
            subsample=best_subsample, colsample_bytree=best_colsample,
            reg_alpha=alpha, reg_lambda=lam,
            scale_pos_weight=spw, tree_method="hist", device="cuda",
            eval_metric="auc", random_state=42
        )
        clf.fit(X_tr, y_tr)

        train_auc = roc_auc_score(y_tr,  clf.predict_proba(X_tr)[:, 1])
        val_auc   = roc_auc_score(y_val, clf.predict_proba(X_val)[:, 1])
        gap = train_auc - val_auc

        mlflow.log_metric("train_auc",   round(train_auc, 5))
        mlflow.log_metric("val_auc",     round(val_auc,   5))
        mlflow.log_metric("overfit_gap", round(gap, 5))

        reg_results.append({"reg_alpha": alpha, "reg_lambda": lam,
                             "train_auc": train_auc, "val_auc": val_auc, "gap": gap})
        print(f"  alpha={alpha} lambda={lam} | Train: {train_auc:.4f} | Val: {val_auc:.4f} | Gap: {gap:.4f}")

reg_df = pd.DataFrame(reg_results)
best_reg = reg_df.loc[reg_df["val_auc"].idxmax()]
best_alpha  = best_reg["reg_alpha"]
best_lambda = best_reg["reg_lambda"]
print(f"\n-> Best reg: alpha={best_alpha}, lambda={best_lambda}")
print(reg_df.to_string())

  alpha=0 lambda=1 | Train: 0.9995 | Val: 0.9657 | Gap: 0.0338
🏃 View run XGB_reg_a0_l1 at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/4/runs/6e149c507e6c488ebbb8d2b86d69c659
🧪 View experiment at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/4
  alpha=0.1 lambda=1 | Train: 0.9994 | Val: 0.9658 | Gap: 0.0337
🏃 View run XGB_reg_a0.1_l1 at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/4/runs/916259dfeb824ae4bc5210f98a9b6f7d
🧪 View experiment at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/4
  alpha=1 lambda=1 | Train: 0.9995 | Val: 0.9663 | Gap: 0.0333
🏃 View run XGB_reg_a1_l1 at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/4/runs/fb3d1acd0d6d40eaab05aca61ccc031d
🧪 View experiment at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/4
  alpha=0 lambda=5 | Train: 0.9992 | Val: 0.9665 | Gap: 0.0327
🏃 View ru

In [14]:
sample_idx = np.random.RandomState(42).choice(len(X_train_final_xgb), size=150_000, replace=False)
X_cv = X_train_final_xgb.iloc[sample_idx]
y_cv = y_train_fe_xgb.iloc[sample_idx]

with mlflow.start_run(run_name="XGB_CrossValidation_5fold"):
    mlflow.log_param("n_estimators",     300)
    mlflow.log_param("max_depth",        best_depth)
    mlflow.log_param("learning_rate",    0.1)
    mlflow.log_param("subsample",        best_subsample)
    mlflow.log_param("colsample_bytree", best_colsample)
    mlflow.log_param("reg_alpha",        best_alpha)
    mlflow.log_param("reg_lambda",       best_lambda)
    mlflow.log_param("scale_pos_weight", spw)
    mlflow.log_param("cv_folds",         5)
    mlflow.log_param("cv_sample_size",   150_000)

    clf_cv = xgb.XGBClassifier(
        n_estimators=300, max_depth=best_depth, learning_rate=0.1,
        subsample=best_subsample, colsample_bytree=best_colsample,
        reg_alpha=best_alpha, reg_lambda=best_lambda,
        scale_pos_weight=spw, tree_method="hist", device="cuda",
        eval_metric="auc", random_state=42
    )

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    cv_scores = cross_val_score(clf_cv, X_cv, y_cv, cv=cv, scoring="roc_auc")

    for i, score in enumerate(cv_scores):
        mlflow.log_metric("fold_auc", round(score, 5), step=i)

    mlflow.log_metric("cv_mean_auc", round(cv_scores.mean(), 5))
    mlflow.log_metric("cv_std_auc",  round(cv_scores.std(),  5))

    print(f"CV folds: {[round(s, 4) for s in cv_scores]}")
    print(f"Mean: {cv_scores.mean():.4f} | Std: {cv_scores.std():.4f}")
    print(f"  -> Std {cv_scores.std():.4f}: {'STABLE' if cv_scores.std() < 0.005 else 'HIGH VARIANCE — check data leakage or small folds'}")

CV folds: [np.float64(0.9375), np.float64(0.9466), np.float64(0.9377), np.float64(0.9396), np.float64(0.9335)]
Mean: 0.9390 | Std: 0.0043
  -> Std 0.0043: STABLE
🏃 View run XGB_CrossValidation_5fold at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/4/runs/5a2ad3b77a6c4f08956bb581ca4274fb
🧪 View experiment at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/4


In [17]:
reg_results = []
for alpha, lam in [(0, 1), (0.1, 1), (1, 1), (0, 5), (1, 5)]:
    label = f"a{alpha}_l{lam}"
    with mlflow.start_run(run_name=f"XGB_reg_{label}"):
        mlflow.log_param("n_estimators",     300)
        mlflow.log_param("max_depth",        best_depth)
        mlflow.log_param("learning_rate",    0.1)
        mlflow.log_param("subsample",        best_subsample)
        mlflow.log_param("colsample_bytree", best_colsample)
        mlflow.log_param("reg_alpha",        alpha)
        mlflow.log_param("reg_lambda",       lam)
        mlflow.log_param("scale_pos_weight", spw)
        mlflow.log_param("tree_method",      "hist")
        mlflow.log_param("device",           "cuda")

        clf = xgb.XGBClassifier(
            n_estimators=300, max_depth=best_depth, learning_rate=0.1,
            subsample=best_subsample, colsample_bytree=best_colsample,
            reg_alpha=alpha, reg_lambda=lam,
            scale_pos_weight=spw, tree_method="hist", device="cuda",
            eval_metric="auc", random_state=42
        )
        clf.fit(X_tr, y_tr)

        train_auc = roc_auc_score(y_tr,  clf.predict_proba(X_tr)[:, 1])
        val_auc   = roc_auc_score(y_val, clf.predict_proba(X_val)[:, 1])
        gap = train_auc - val_auc

        mlflow.log_metric("train_auc",   round(train_auc, 5))
        mlflow.log_metric("val_auc",     round(val_auc,   5))
        mlflow.log_metric("overfit_gap", round(gap, 5))

        reg_results.append({"reg_alpha": alpha, "reg_lambda": lam,
                             "train_auc": train_auc, "val_auc": val_auc, "gap": gap})
        print(f"  alpha={alpha} lambda={lam} | Train: {train_auc:.4f} | Val: {val_auc:.4f} | Gap: {gap:.4f}")

reg_df = pd.DataFrame(reg_results)
best_reg = reg_df.loc[reg_df["val_auc"].idxmax()]
best_alpha  = best_reg["reg_alpha"]
best_lambda = best_reg["reg_lambda"]
print(f"\n-> Best reg: alpha={best_alpha}, lambda={best_lambda}")
print(reg_df.to_string())

  alpha=0 lambda=1 | Train: 0.9995 | Val: 0.9657 | Gap: 0.0338
🏃 View run XGB_reg_a0_l1 at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/4/runs/b8c0b30c21e144caa6f247a97eb72bfb
🧪 View experiment at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/4
  alpha=0.1 lambda=1 | Train: 0.9994 | Val: 0.9658 | Gap: 0.0337
🏃 View run XGB_reg_a0.1_l1 at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/4/runs/e84fc5171a9f4b9e917d123927b6bb4a
🧪 View experiment at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/4
  alpha=1 lambda=1 | Train: 0.9995 | Val: 0.9663 | Gap: 0.0333
🏃 View run XGB_reg_a1_l1 at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/4/runs/83fa37ca62304e8f96b50fa4de541cab
🧪 View experiment at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/4
  alpha=0 lambda=5 | Train: 0.9992 | Val: 0.9665 | Gap: 0.0327
🏃 View ru

In [18]:
import pickle
from sklearn.pipeline import Pipeline

X_tr_f, X_val_f, y_tr_f, y_val_f = train_test_split(
    X_train_final_xgb, y_train_fe_xgb,
    test_size=0.2, random_state=42, stratify=y_train_fe_xgb
)

neg_f = int((y_tr_f == 0).sum())
pos_f = int((y_tr_f == 1).sum())
spw_f = round(neg_f / pos_f, 2)

with mlflow.start_run(run_name="XGB_Final_Pipeline") as run:
    mlflow.log_param("n_estimators",     500)
    mlflow.log_param("max_depth",        best_depth)
    mlflow.log_param("learning_rate",    0.05)
    mlflow.log_param("subsample",        best_subsample)
    mlflow.log_param("colsample_bytree", best_colsample)
    mlflow.log_param("reg_alpha",        best_alpha)
    mlflow.log_param("reg_lambda",       best_lambda)
    mlflow.log_param("scale_pos_weight", spw_f)
    mlflow.log_param("tree_method",      "hist")
    mlflow.log_param("device",           "cuda")
    mlflow.log_param("note",             "lr=0.05, 500 trees, best params from all sweeps")

    final_clf = xgb.XGBClassifier(
        n_estimators=500, max_depth=best_depth, learning_rate=0.05,
        subsample=best_subsample, colsample_bytree=best_colsample,
        reg_alpha=best_alpha, reg_lambda=best_lambda,
        scale_pos_weight=spw_f, tree_method="hist", device="cuda",
        eval_metric="auc", random_state=42
    )
    final_clf.fit(X_tr_f, y_tr_f)

    val_auc   = roc_auc_score(y_val_f, final_clf.predict_proba(X_val_f)[:, 1])
    train_auc = roc_auc_score(y_tr_f,  final_clf.predict_proba(X_tr_f)[:, 1])
    gap = train_auc - val_auc

    mlflow.log_metric("train_auc",   round(train_auc, 5))
    mlflow.log_metric("val_auc",     round(val_auc,   5))
    mlflow.log_metric("overfit_gap", round(gap, 5))

    final_pipe = Pipeline([("clf", final_clf)])

    mlflow.sklearn.log_model(
        sk_model=final_pipe,
        artifact_path="xgboost_pipeline",
        registered_model_name="XGBoost_FraudDetection"
    )

    # save pkl for local use
    pkl_path = "/kaggle/working/xgboost_final_pipeline.pkl"
    with open(pkl_path, "wb") as f:
        pickle.dump(final_pipe, f)
    mlflow.log_artifact(pkl_path)

    print(f"[Final] Train: {train_auc:.4f} | Val: {val_auc:.4f} | Gap: {gap:.4f}")
    print(f"  -> {'OVERFIT' if gap > 0.03 else 'OK'}")
    print(f"Run ID: {run.info.run_id}")
    print("✅ Model registered as XGBoost_FraudDetection")

2026/05/02 19:04:49 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/02 19:04:49 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
Registered model 'XGBoost_FraudDetection' already exists. Creating a new version of this model...
2026/05/02 19:04:58 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: XGBoost_FraudDetection, version 4
Created version '4' of model 'XGBoost_FraudDetection'.


[Final] Train: 0.9986 | Val: 0.9672 | Gap: 0.0314
  -> OVERFIT
Run ID: cd0a105927694771b66453f103b591ab
✅ Model registered as XGBoost_FraudDetection
🏃 View run XGB_Final_Pipeline at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/4/runs/cd0a105927694771b66453f103b591ab
🧪 View experiment at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/4
